# 05 — Streaming Responses

Three streaming patterns: direct LLM, chain, and event streaming.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage

## Example 1: Direct LLM Streaming

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

for chunk in llm.stream([HumanMessage(content="Write a haiku about programming.")]):
    print(chunk.content, end="", flush=True)
print()

## Example 2: Chain Streaming

In [ ]:
prompt = ChatPromptTemplate.from_template("Explain {topic} in exactly 3 bullet points. Be concise.")
chain = prompt | llm | StrOutputParser()

for chunk in chain.stream({"topic": "how neural networks learn"}):
    print(chunk, end="", flush=True)
print()

## Example 3: Async Stream Events

In [ ]:
prompt = ChatPromptTemplate.from_template("List 3 fun facts about {topic}.")
chain = prompt | llm | StrOutputParser()

async for event in chain.astream_events({"topic": "octopuses"}, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            print(content, end="", flush=True)
    elif kind == "on_chain_start":
        print(f"  [Chain started: {event['name']}]")
    elif kind == "on_chain_end" and event["name"] == "RunnableSequence":
        print(f"\n  [Chain finished]")